# Optimización de hiperparámetros MDLF con Optuna — v3 LINE

**Datos**: almacén `50mil_tarjetas_3_al_31_marzo`  
**Split**: temporal — train: 3-23 marzo / test: 24-31 marzo  
**Estrategia**: 80 trials × 5 épocas (proxy rápido), luego reentrenamiento con 30 épocas de la mejor config  
**Objetivo**: maximizar Top-1 accuracy en test

### Hiperparámetros: fijos vs variables
Fijos = **100% consistentes en el top-15 de v2**, o demostrados rotos.

| Grupo | Parámetro | Valor fijo | Razón |
|---|---|---|---|
| LINE | lr, fo_epochs, so_epochs, init_scale | 0.025 / 200 / 1000 / 0.1 | Estándar LINE |
| LDA | lda_alpha_none | False | 15/15 top-v2 usan alpha explícita |
| LDA | lda_token_granularity | 'temporal' | 15/15 top-v2 |
| LDA | lda_use_periods | False | 15/15 top-v2 |
| CNN | kernel_size | 3 | Domina en top-v2 |
| Train | lr_scheduler | True | 15/15 top-v2 |
| Ext. | include_prox | True | 15/15 top-v2 |

| Grupo | Parámetros variables |
|---|---|
| LINE | dim [16,32,64], asymmetric, num_negative [1,5] |
| LDA | n_topics, lda_alpha, lda_beta_none/beta, lda_log_counts, lda_min_df, lda_min_station_trips |
| CNN | n_heads, d_t, cnn_channels |
| Train | optimizer, lr, weight_decay, n_weeks |
| Ext. | include_neighborhood_crowd |
| Norm | norm [l2, None] |

In [ ]:
import os, sys, pickle
from pathlib import Path

_cubo_dir = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo').resolve()
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF').resolve()

os.chdir(_cubo_dir)
for _p in [str(_cubo_dir), str(_mdlf_dir), str(_mdlf_dir / 'LINE')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"CWD → {os.getcwd()}")

import optuna
import pandas as pd
import json
from datetime import datetime
from pipeline_mdlf import PipelineMDLF
from station_line import StationLINE

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ── Configuración fija ────────────────────────────────────────────────────────

ALMACEN      = '50mil_tarjetas_3_al_31_marzo'
SPLIT_DATE   = '2025-03-24'
N_EPOCHS_OPTUNA = 7    # proxy más fiable que 5, reduce ruido en el ranking
N_EPOCHS_FINAL  = 30
N_TRIALS        = 150  # ~7-8h overnight (≈3 min/trial: 30s LINE + 2.5min MDLF)

# v3: LINE en Optuna + fijados los params 100% consistentes en top-15 de v2
STUDY_NAME = 'mdlf_marzo2025_v3_line'
DB_PATH    = _mdlf_dir / 'runs' / f'{STUDY_NAME}.db'

# Grafo de cercanías — se carga una vez, se reutiliza en cada trial
_GRAPH_PATH = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo/check_transbordo/cercanias_multidigraph_sin_civis.pkl')
with open(_GRAPH_PATH, 'rb') as _f:
    G_cercanias = pickle.load(_f)
print(f"Grafo cercanías: {G_cercanias.number_of_nodes()} nodos | {G_cercanias.number_of_edges()} aristas")

_descomp = Path('/home/jupyter-lquijano/david/estudios/descomprimido')
TTP_PATHS = [
    str(_descomp / 'TTP_PERIODO_01_2025.csv'),
    str(_descomp / 'TTP_PERIODO_02_2025.csv'),
    str(_descomp / 'TTP_PERIODO_03_2025.csv'),
]

OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero',
    'validaciones_8_9_febrero',
    'validaciones_10_16_febrero',
    'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

print(f"Almacén  : {ALMACEN}")
print(f"Split    : train < {SPLIT_DATE} | test >= {SPLIT_DATE}")
print(f"Trials   : {N_TRIALS} × {N_EPOCHS_OPTUNA} épocas  (~7-8h overnight)")
print(f"Study DB : {DB_PATH}")

In [ ]:
# ── Función objetivo ─────────────────────────────────────────────────────────

def objective(trial: optuna.Trial) -> float:

    # ── LINE — variable: dim, asymmetric, num_negative ───────────────────────
    # Fijos: lr=0.025, fo=200, so=1000, init_scale=0.1 (estándar LINE)
    line_dim          = trial.suggest_categorical('line_dim', [16, 32, 64])
    line_asymmetric   = trial.suggest_categorical('line_asymmetric', [True, False])
    line_num_negative = trial.suggest_categorical('line_num_negative', [1, 5])

    line_model = StationLINE(
        dim=line_dim,
        num_negative=line_num_negative,
        lr=0.025,
        num_epochs_fo=200,
        num_epochs_so=1000,
        init_scale=0.1,
        asymmetric=line_asymmetric,
        random_state=42,
    )
    _emb_station = line_model.fit_transform(G_cercanias)
    emb_ap       = line_model.to_punto_acceso(G_cercanias)

    # Parquet temporal — se elimina siempre en el bloque finally
    _tmp_line = _mdlf_dir / 'runs' / f'line_trial_{trial.number:03d}.parquet'
    emb_ap.to_parquet(_tmp_line)

    try:
        # ── LDA ──────────────────────────────────────────────────────────────
        # Fijo: lda_alpha_none=False (15/15 top-v2), token_granularity='temporal', use_periods=False
        n_topics = trial.suggest_categorical('n_topics', [8, 16, 32, 48])
        lda_alpha = trial.suggest_float('lda_alpha', 0.001, 1.0, log=True)

        lda_beta_none = trial.suggest_categorical('lda_beta_none', [True, False])
        lda_beta = None if lda_beta_none else trial.suggest_float('lda_beta', 0.001, 1.0, log=True)

        lda_log_counts        = trial.suggest_categorical('lda_log_counts', [True, False])
        lda_min_df            = trial.suggest_int('lda_min_df', 1, 5)
        lda_min_station_trips = trial.suggest_categorical('lda_min_station_trips', [0, 50, 100])

        # ── CNN ──────────────────────────────────────────────────────────────
        # Fijo: kernel_size=3 (domina en top-v2)
        n_heads      = trial.suggest_categorical('n_heads', [2, 4, 8])
        d_t          = trial.suggest_categorical('d_t', [8, 16, 32])
        cnn_channels = trial.suggest_categorical('cnn_channels', [8, 16, 32])

        # ── Entrenamiento ─────────────────────────────────────────────────────
        # Fijo: lr_scheduler=True (15/15 top-v2)
        optimizer    = trial.suggest_categorical('optimizer', ['sgd', 'adamw'])
        lr           = trial.suggest_float('lr', 1e-3, 1e-1, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True)
        n_weeks      = trial.suggest_int('n_weeks', 1, 4)

        # ── Extensiones ───────────────────────────────────────────────────────
        # Fijo: include_prox=True (15/15 top-v2)
        include_neighborhood_crowd = trial.suggest_categorical('include_neighborhood_crowd', [True, False])

        # ── Normalización ─────────────────────────────────────────────────────
        norm_str = trial.suggest_categorical('norm', ['l2', 'none'])
        norm = None if norm_str == 'none' else norm_str

        # ── Lanzar pipeline ───────────────────────────────────────────────────
        pipe = PipelineMDLF(
            nombre_almacen=ALMACEN,
            ttp_paths=TTP_PATHS,
            od_matrices_subdirs=OD_MATRICES_SUBDIRS,
            line_ap_path=_tmp_line,
            split_date=SPLIT_DATE,
            n_topics_cercanias=n_topics,
            lda_alpha=lda_alpha,
            lda_beta=lda_beta,
            lda_use_periods=False,             # fijo: 15/15 top-v2
            lda_min_df=lda_min_df,
            lda_log_counts=lda_log_counts,
            lda_token_granularity='temporal',  # fijo: 15/15 top-v2
            lda_min_station_trips=lda_min_station_trips,
            n_heads=n_heads,
            d_t=d_t,
            cnn_channels=cnn_channels,
            kernel_size=3,                     # fijo: domina en top-v2
            optimizer=optimizer,
            lr=lr,
            weight_decay=weight_decay,
            lr_scheduler=True,                 # fijo: 15/15 top-v2
            n_weeks=n_weeks,
            include_prox=True,                 # fijo: 15/15 top-v2
            td_mode='context',                 # fijo: paper + moda top-v2
            include_neighborhood_crowd=include_neighborhood_crowd,
            norm=norm,
            n_epochs=N_EPOCHS_OPTUNA,
            random_state=42,
            line_meta={
                'dim': line_dim, 'asymmetric': line_asymmetric,
                'num_negative': line_num_negative,
                'lr': 0.025, 'num_epochs_fo': 200, 'num_epochs_so': 1000,
            },
        )

        pipe.run(run_name=f'optuna_v3_t{trial.number:03d}')
        trial.set_user_attr('top5', pipe.test_accuracy5)  # guardado en DB
        return pipe.test_accuracy

    finally:
        _tmp_line.unlink(missing_ok=True)

In [ ]:
# ── Ejecutar estudio Optuna ───────────────────────────────────────────────────
# load_if_exists=True → si el .db ya existe, reanuda desde donde se dejó

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=40),
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=True,
)

trials_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"Study    : {STUDY_NAME}")
print(f"DB       : {DB_PATH}")
print(f"Trials   : {trials_done} completados / {N_TRIALS} objetivo")
print()

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

print(f"\n✓ Estudio completado")
print(f"  Mejor Top-1 accuracy : {study.best_value:.4f}")
print(f"  Trial número         : {study.best_trial.number}")

In [ ]:
# ── Resultados ────────────────────────────────────────────────────────────────

print("=" * 60)
print("MEJORES HIPERPARÁMETROS")
print("=" * 60)
for k, v in study.best_trial.params.items():
    print(f"  {k:<40} {v}")
print(f"  {'Top-1 accuracy (5 épocas)':<40} {study.best_value:.4f}")

df_trials = study.trials_dataframe()
df_trials = df_trials.sort_values('value', ascending=False)
print(f"\nTop-10 trials:")
display(df_trials[['number', 'value', 'duration']].head(10))

In [ ]:
# ── Importancia de hiperparámetros ────────────────────────────────────────────
try:
    importances = optuna.importance.get_param_importances(study)
    print("Importancia de hiperparámetros (fANOVA):")
    for param, imp in importances.items():
        bar = '█' * int(imp * 50)
        print(f"  {param:<40} {imp:.4f}  {bar}")
except Exception as e:
    print(f"No se pudo calcular importancia: {e}")

In [ ]:
# ── Reentrenamiento con mejor configuración y épocas completas ────────────────

best = study.best_trial.params

_lda_beta = None if best.get('lda_beta_none', True) else best.get('lda_beta')
_norm     = None if best.get('norm', 'l2') == 'none' else best.get('norm', 'l2')

# Entrenar LINE con la mejor config LINE
print(f"Entrenando LINE (dim={best['line_dim']}, asymmetric={best['line_asymmetric']}, num_negative={best['line_num_negative']})...")
line_best = StationLINE(
    dim=best['line_dim'],
    num_negative=best['line_num_negative'],
    lr=0.025,
    num_epochs_fo=200,
    num_epochs_so=1000,
    init_scale=0.1,
    asymmetric=best['line_asymmetric'],
    random_state=42,
)
_emb_station_best = line_best.fit_transform(G_cercanias)
emb_ap_best       = line_best.to_punto_acceso(G_cercanias)

_line_best_path = _mdlf_dir / 'runs' / f'{STUDY_NAME}_line_best.parquet'
emb_ap_best.to_parquet(_line_best_path)
print(f"LINE best guardado: {_line_best_path.name}  ({emb_ap_best.shape[1]} dims)")

print(f"\nReentrenando MDLF con {N_EPOCHS_FINAL} épocas...")
print()

pipe_best = PipelineMDLF(
    nombre_almacen=ALMACEN,
    ttp_paths=TTP_PATHS,
    od_matrices_subdirs=OD_MATRICES_SUBDIRS,
    line_ap_path=_line_best_path,
    split_date=SPLIT_DATE,
    n_topics_cercanias=best['n_topics'],
    lda_alpha=best['lda_alpha'],
    lda_beta=_lda_beta,
    lda_use_periods=False,
    lda_min_df=best['lda_min_df'],
    lda_log_counts=best['lda_log_counts'],
    lda_token_granularity='temporal',
    lda_min_station_trips=best['lda_min_station_trips'],
    n_heads=best['n_heads'],
    d_t=best['d_t'],
    cnn_channels=best['cnn_channels'],
    kernel_size=3,
    optimizer=best['optimizer'],
    lr=best['lr'],
    weight_decay=best['weight_decay'],
    lr_scheduler=True,
    n_weeks=best['n_weeks'],
    include_prox=True,
    td_mode='context',
    include_neighborhood_crowd=best['include_neighborhood_crowd'],
    norm=_norm,
    n_epochs=N_EPOCHS_FINAL,
    random_state=42,
    line_meta={
        'dim': best['line_dim'], 'asymmetric': best['line_asymmetric'],
        'num_negative': best['line_num_negative'],
        'lr': 0.025, 'num_epochs_fo': 200, 'num_epochs_so': 1000,
    },
)

pipe_best.run(run_name='optuna_v3_best_final')

print(f"\n{'='*50}")
print(f"RESULTADO FINAL (best config, {N_EPOCHS_FINAL} épocas)")
print(f"  Top-1 accuracy : {pipe_best.test_accuracy:.4f}")
print(f"  Top-5 accuracy : {pipe_best.test_accuracy5:.4f}")
print(f"  (Proxy 5 épocas era: {study.best_value:.4f})")

In [ ]:
# ── Guardar mejores params en JSON ────────────────────────────────────────────

_out = {
    'study_name': STUDY_NAME,
    'best_trial': study.best_trial.number,
    'top1_proxy_5epochs': study.best_value,
    'top1_final_30epochs': pipe_best.test_accuracy,
    'top5_final_30epochs': pipe_best.test_accuracy5,
    'line_dims_final': int(best['line_dim']) * 2,
    'best_params': best,
    'fixed_params': {
        'include_prox': True,
        'td_mode': 'context',
        'lda_alpha_none': False,
        'lda_use_periods': False,
        'lda_token_granularity': 'temporal',
        'lr_scheduler': True,
        'kernel_size': 3,
        'line_lr': 0.025,
        'line_num_epochs_fo': 200,
        'line_num_epochs_so': 1000,
        'line_init_scale': 0.1,
    },
}

_json_path = _mdlf_dir / 'runs' / f'{STUDY_NAME}_best.json'
with open(_json_path, 'w') as f:
    json.dump(_out, f, indent=2, default=str)

print(f"Resultados guardados en: {_json_path}")

In [ ]:
# ── Top combinaciones de hiperparámetros ──────────────────────────────────────

import pandas as pd

df_trials = study.trials_dataframe()
df_ok = df_trials[df_trials['state'] == 'COMPLETE'].copy()

param_cols = [c for c in df_ok.columns if c.startswith('params_')]
param_display = {c: c.replace('params_', '') for c in param_cols}

top_n = 15
df_top = (
    df_ok
    .sort_values('value', ascending=False)
    .head(top_n)
    [['number', 'value', 'duration'] + param_cols]
    .rename(columns={'number': 'trial', 'value': 'top1_proxy', **param_display})
    .reset_index(drop=True)
)

print(f'Top {top_n} trials (de {len(df_ok)} completados) — proxy 5 épocas')
print(f'Mejor Top-1 proxy : {df_ok["value"].max():.4f}')
print(f'Mediana           : {df_ok["value"].median():.4f}')
print()
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 10 else f'{x:.2f}')
display(df_top)

# ── Tabla compacta — parámetros clave ────────────────────────────────────────
key_params = [
    'line_dim', 'line_asymmetric', 'line_num_negative',
    'n_topics', 'lda_alpha', 'lda_log_counts', 'lda_min_df',
    'n_heads', 'd_t', 'cnn_channels',
    'optimizer', 'lr', 'n_weeks', 'include_neighborhood_crowd',
]
available = ['trial', 'top1_proxy'] + [p for p in key_params if p in df_top.columns]

print()
print('── Parámetros clave ──────────────────────────────────────────────────────')
display(df_top[available])

## Top 15 combinaciones — v3 (carga directa desde DB)

In [ ]:
# ── Cargar estudio desde DB (standalone) ──────────────────────────────────────
import optuna, pandas as pd, warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pathlib import Path
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF')
_db = _mdlf_dir / 'runs' / 'mdlf_marzo2025_v3_line.db'

_study = optuna.load_study(
    study_name='mdlf_marzo2025_v3_line',
    storage=f'sqlite:///{_db}',
)

_df = _study.trials_dataframe()
_df_ok = _df[_df['state'] == 'COMPLETE'].copy()

# Renombrar columnas params_ → nombre limpio
_param_cols = [c for c in _df_ok.columns if c.startswith('params_')]
_df_ok = _df_ok.rename(columns={c: c.replace('params_', '') for c in _param_cols})
# user_attrs_top5 → top5 (solo existe en trials ejecutados tras añadir set_user_attr)
if 'user_attrs_top5' in _df_ok.columns:
    _df_ok = _df_ok.rename(columns={'user_attrs_top5': 'top5'})
_df_ok = _df_ok.rename(columns={'number': 'trial', 'value': 'top1'})

TOP_N = 15
_df_top15 = (
    _df_ok
    .sort_values('top1', ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)
_df_top15.index += 1  # rank 1-based

_has_top5 = 'top5' in _df_top15.columns

print(f"Study  : mdlf_marzo2025_v3_line")
print(f"Trials : {len(_df_ok)} completados de {len(_df)} total  (7 épocas proxy)")
print(f"Mejor  : top1={_df_ok['top1'].max():.4f}" + (f"  top5={_df_ok['top5'].max():.4f}" if _has_top5 else "  (top5 no disponible en trials anteriores)") + f"  (trial #{int(_df_top15.loc[1,'trial'])})")
print(f"Mediana top1: {_df_ok['top1'].median():.4f}")

In [ ]:
# ── Top 15 — todos los hiperparámetros, agrupados por sección ─────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 20)

_score_cols = ['trial', 'top1'] + (['top5'] if _has_top5 else [])
_base = _df_top15[_score_cols].copy()
_fmt_base = {'top1': '{:.4f}', **({'top5': '{:.4f}'} if _has_top5 else {})}

# ── LINE ──────────────────────────────────────────────────────────────────────
_line_cols = ['line_dim', 'line_asymmetric', 'line_num_negative']
print("━" * 80)
print(f"  TOP {TOP_N}  —  LINE")
print("━" * 80)
display(_base.join(_df_top15[_line_cols]).style.format(_fmt_base))

# ── LDA ───────────────────────────────────────────────────────────────────────
_lda_cols = ['n_topics', 'lda_alpha', 'lda_beta_none', 'lda_beta',
             'lda_log_counts', 'lda_min_df', 'lda_min_station_trips']
_lda_cols = [c for c in _lda_cols if c in _df_top15.columns]
print()
print("━" * 80)
print(f"  TOP {TOP_N}  —  LDA")
print("━" * 80)
display(_base.join(_df_top15[_lda_cols]).style.format({
    **_fmt_base,
    'lda_alpha': '{:.5f}',
    'lda_beta':  lambda x: f'{x:.5f}' if pd.notna(x) else '—',
}))

# ── CNN + Train + Extras ───────────────────────────────────────────────────────
_rest_cols = ['n_heads', 'd_t', 'cnn_channels',
              'optimizer', 'lr', 'weight_decay', 'n_weeks',
              'include_neighborhood_crowd', 'norm']
_rest_cols = [c for c in _rest_cols if c in _df_top15.columns]
print()
print("━" * 80)
print(f"  TOP {TOP_N}  —  CNN + Train + Extras")
print("━" * 80)
display(_base.join(_df_top15[_rest_cols]).style.format({
    **_fmt_base,
    'lr':           '{:.5f}',
    'weight_decay': '{:.5f}',
}))